# Fabric Admin Toolkit - Bootstrap Deployment Notebook

This notebook deploys the standard Wipfli MAC Tech Delivery admin toolkit 
to a target Fabric workspace. It pulls artifact definitions directly from 
the GitHub repo and uses the `fabric-cicd` library to create or update 
items in the target workspace.

## Prerequisites
- The target workspace must already exist
- The user running this notebook must be a workspace Admin or Contributor 
  on the target workspace
- A GitHub PAT with read access to the `fabric-admin-toolkit` repo must 
  be available
- A `parameter.yml` file must exist at the repo root (even if empty)

## What Gets Deployed
- Notebooks
- Data Pipelines
- Lakehouses
- Environments

## Notes
- If an item already exists in the target workspace by name, it will be 
  updated. If it does not exist, it will be created.
- Pipelines with connections require the target workspace connection GUIDs 
  to be configured in `parameter.yml` before deployment.
- Lakehouses are provisioned as empty containers — no data is migrated.
- OAuth-bound activities (Office365Email, Teams) cannot be deployed via SP.
  Remove these activities from pipeline definitions before deploying and
  add them manually in the target workspace after deployment.

## Step 1: Install Dependencies

Installs `fabric-cicd` into the current notebook session.
Pinned to a specific version to ensure consistent behavior across deployments.
Update the version pin deliberately when upgrading — do not use `--upgrade` 
or omit the version, as new releases may introduce breaking API changes.

In [ ]:
%pip install fabric-cicd==0.3.1 --quiet

## Step 2: Configure Deployment Parameters

Update the variables in this cell before running the notebook:

- `GITHUB_PAT` — fine-grained token with read access to the Wipfli repo.
  Store this securely — do not commit this notebook with a real token 
  value checked in. This is a Wipfli PAT authenticating to the Wipfli repo
  and should not be stored in the client Key Vault.
- `TARGET_WORKSPACE_ID` — the GUID of the workspace you are deploying 
  to. Found in the workspace URL in Fabric.
- `ENVIRONMENT` — must match a key defined in `parameter.yml` in the 
  repo. Used to substitute environment-specific GUIDs (connections, 
  lakehouses, etc.) during deployment.
- `ITEM_TYPES_IN_SCOPE` — controls which artifact types get deployed. 
  Comment out any types you want to skip for a given run.

In [ ]:
# GitHub config — Wipfli repo, PAT entered directly (not stored in client KV)
GITHUB_PAT    = ""  # do not commit with a real token value
GITHUB_ORG    = "Wipfli-MAC-TechDelivery"
GITHUB_REPO   = "fabric-admin-toolkit"
GITHUB_BRANCH = "main"

# Target workspace config
TARGET_WORKSPACE_ID = ""  # client workspace GUID from workspace URL
ENVIRONMENT         = "CLIENT"  # must match keys in parameter.yml

# Artifact types to deploy — comment out any types to skip for a given run
ITEM_TYPES_IN_SCOPE = [
    "Notebook",
    "DataPipeline",
    "Lakehouse",
    "Environment"
]

## Step 3: Download Repo from GitHub

Downloads the entire repo as a zip file from GitHub and extracts it to 
ephemeral session storage at `/tmp/admin-toolkit`. 

A few things to note:
- `/tmp` is session-scoped — it gets wiped when the notebook session ends, 
  so there is no cleanup needed and no sensitive content persists
- GitHub zips extract into a dynamically named subdirectory 
  (`org-repo-commithash/`) so we locate the root directory programmatically 
  rather than hardcoding the path
- `resp.raise_for_status()` will throw an error early if the PAT is invalid 
  or the repo is not accessible, before any deployment logic runs

In [ ]:
import requests
import zipfile
import io
import os
import glob

REPO_LOCAL_PATH = "/tmp/admin-toolkit"

def download_repo(org, repo, branch, pat, extract_path):
    url = f"https://api.github.com/repos/{org}/{repo}/zipball/{branch}"
    headers = {
        "Authorization": f"Bearer {pat}",
        "Accept": "application/vnd.github+json"
    }
    
    print(f"Downloading {org}/{repo} @ {branch}...")
    resp = requests.get(url, headers=headers)
    resp.raise_for_status()
    
    z = zipfile.ZipFile(io.BytesIO(resp.content))
    z.extractall(extract_path)
    
    # GitHub zips extract into a subdirectory named "org-repo-commithash/"
    # so we locate it dynamically rather than hardcoding the path
    extracted_dirs = [
        d for d in glob.glob(f"{extract_path}/*/")
        if os.path.isdir(d)
    ]
    
    if not extracted_dirs:
        raise Exception("Could not find extracted repo directory")
    
    repo_root = extracted_dirs[0]
    print(f"Extracted to: {repo_root}")
    return repo_root

repo_path = download_repo(
    GITHUB_ORG,
    GITHUB_REPO,
    GITHUB_BRANCH,
    GITHUB_PAT,
    REPO_LOCAL_PATH
)

## Step 4: Deploy Notebooks

Deploys notebooks first since pipelines reference them by ID. 
We need the notebook IDs in the target workspace before we can 
wire up the pipeline definitions.

In [ ]:
from fabric_cicd import FabricWorkspace, publish_all_items

# Deploy notebooks first
notebook_workspace = FabricWorkspace(
    workspace_id=TARGET_WORKSPACE_ID,
    environment=ENVIRONMENT,
    repository_directory=repo_path,
    item_type_in_scope=["Notebook"]
)

publish_all_items(notebook_workspace)
print("Notebooks deployed.")

## Step 5: Look Up Deployed Notebook IDs

After deploying notebooks, we query the Fabric Items API to get the 
IDs of the notebooks that now exist in the target workspace. These IDs 
are workspace-specific and weren't known ahead of time, so we look them 
up dynamically rather than hardcoding them.

This is necessary because pipelines reference notebooks by ID in their 
definition, and those IDs differ between workspaces.

In [ ]:
import requests

token = notebookutils.credentials.getToken("https://api.fabric.microsoft.com")
headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
}

def get_workspace_items(workspace_id, item_type):
    """Returns a dict of {displayName: id} for all items of a given type"""
    url = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items?type={item_type}"
    resp = requests.get(url, headers=headers)
    resp.raise_for_status()
    return {item["displayName"]: item["id"] for item in resp.json().get("value", [])}

notebooks = get_workspace_items(TARGET_WORKSPACE_ID, "Notebook")
print("Notebooks found in target workspace:")
for name, item_id in notebooks.items():
    print(f"  {name}: {item_id}")

## Step 6: Generate parameter.yml Dynamically

Builds the parameter.yml substitution file in memory using the notebook 
IDs we just looked up, then writes it to /tmp so fabric-cicd can read it 
during pipeline deployment.

Uses `$workspace.id` as the dynamic variable for workspace ID substitution
so the target workspace ID is resolved at deploy time by fabric-cicd rather
than hardcoded — no need to pass TARGET_WORKSPACE_ID into the substitution.

Update `NOTEBOOK_REFERENCES` to map each notebook name to the source ID 
embedded in the pipeline definitions in the repo. Source IDs come from 
the pipeline JSON — the notebookId values.

In [ ]:
import os

# Map notebook display names to their source IDs in the pipeline definitions
# Source IDs come from the notebookId values in the pipeline JSON in the repo
NOTEBOOK_REFERENCES = {
    "nb_admin__weekly_vacuum_maintenance": "3352cbc7-3b0a-42a5-9075-529c833d7b30"
}

# Source workspace ID embedded in pipeline definitions in the Wipfli repo
SOURCE_WORKSPACE_ID = "e3898012-c205-4a95-91a3-bcec2def2a30"

lines = ["find_replace:"]

# Substitute source workspace ID — $workspace.id resolves dynamically at deploy time
lines.append(f'  - find_value: "{SOURCE_WORKSPACE_ID}"')
lines.append(f'    replace_value:')
lines.append(f'      {ENVIRONMENT}: "$workspace.id"')

# Substitute each notebook ID with the ID deployed in the target workspace
for notebook_name, source_id in NOTEBOOK_REFERENCES.items():
    if notebook_name not in notebooks:
        raise Exception(
            f"Notebook '{notebook_name}' not found in target workspace. "
            f"Check display name matches exactly."
        )
    target_id = notebooks[notebook_name]
    lines.append(f'  - find_value: "{source_id}"')
    lines.append(f'    replace_value:')
    lines.append(f'      {ENVIRONMENT}: "{target_id}"')

param_content = "\n".join(lines)
param_file_path = "/tmp/parameter.yml"

with open(param_file_path, "w") as f:
    f.write(param_content)

print("Generated parameter.yml:")
print(param_content)

## Step 7: Deploy Remaining Artifacts

Deploys pipelines, lakehouses, and environments using the dynamically 
generated parameter.yml. fabric-cicd applies the substitutions built 
in the previous step before pushing pipeline definitions to the Fabric API,
ensuring all notebook and workspace ID references point to the correct 
resources in the target workspace.

Note: the `parameter` argument accepts a file path to the parameter.yml.
fabric-cicd reads and applies it automatically during deployment.

In [ ]:
from fabric_cicd import FabricWorkspace, publish_all_items

pipeline_workspace = FabricWorkspace(
    workspace_id=TARGET_WORKSPACE_ID,
    environment=ENVIRONMENT,
    repository_directory=repo_path,
    item_type_in_scope=["DataPipeline", "Lakehouse", "Environment"],
    parameter=param_file_path
)

publish_all_items(pipeline_workspace)
print("Deployment complete.")